# 01 — Exploratory Data Analysis

Explore the raw scraped corpus before annotation.

In [2]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

ModuleNotFoundError: No module named 'pandas'

In [2]:
import sys
sys.path.append("..")

In [4]:
import requests

article_id = "48523909"
url = f"https://content.core.api.espn.com/v1/sports/news/{article_id}"
r = requests.get(url)
print(r.status_code)
print(list(r.json().keys()))  # see what keys actually exist

200
['resultsCount', 'resultsLimit', 'resultsOffset', 'headlines', 'breakingNews', 'timestamp', 'status']


In [5]:
import requests

article_id = "48523909"
url = f"https://content.core.api.espn.com/v1/sports/news/{article_id}"
r = requests.get(url)
data = r.json()

headlines = data.get("headlines", [])
print(f"Headlines count: {len(headlines)}")
if headlines:
    print("Keys in first headline:", list(headlines[0].keys()))

Headlines count: 1
Keys in first headline: ['id', 'nowId', 'contentKey', 'dataSourceIdentifier', 'publishedkey', 'type', 'gameId', 'headline', 'description', 'title', 'linkText', 'categorized', 'originallyPosted', 'lastModified', 'published', 'root', 'section', 'source', 'images', 'video', 'categories', 'keywords', 'story', 'premium', 'isLiveBlog', 'links', 'allowSearch', 'allowContentReactions']


In [5]:
import logging
from src.data_pipeline.scraper import SportsScraper

logging.basicConfig(level=logging.INFO)

# Clear progress first so we get a fresh run
import json
from pathlib import Path
progress_file = Path("data/raw/progress.json")
if progress_file.exists():
    progress_file.unlink()
    print("Cleared progress.json")

scraper = SportsScraper(output_dir="data/raw", delay_seconds=1.0)
articles = scraper.scrape_espn_articles(sport="nba", max_articles=50)

print(f"\nTotal articles: {len(articles)}")
print(f"Full body (>300 chars): {sum(1 for a in articles if len(a['body']) > 300)}")
print(f"Fell back to description: {sum(1 for a in articles if a['body'] == a['description'])}")
print(f"Avg body length: {sum(len(a['body']) for a in articles) // len(articles)} chars")

INFO:src.data_pipeline.scraper:Fetching ESPN NBA article list from https://site.api.espn.com/apis/site/v2/sports/basketball/nba/news?limit=50


Cleared progress.json


INFO:src.data_pipeline.scraper:Found 50 articles in API response
INFO:src.data_pipeline.scraper:Collected 43 articles for nba



Total articles: 43
Full body (>300 chars): 43
Fell back to description: 0
Avg body length: 8852 chars


In [6]:
import logging
from src.data_pipeline.scraper import SportsScraper

logging.basicConfig(level=logging.INFO)

scraper = SportsScraper(output_dir="data/raw", delay_seconds=1.0)
scraper.run(sports=["nba", "nfl", "mlb", "nhl"], max_per_source=50)

INFO:src.data_pipeline.scraper:--- Starting NBA ---
INFO:src.data_pipeline.scraper:Fetching ESPN NBA article list from https://site.api.espn.com/apis/site/v2/sports/basketball/nba/news?limit=50
INFO:src.data_pipeline.scraper:Found 50 articles in API response
INFO:src.data_pipeline.scraper:Collected 0 articles for nba
ERROR:src.data_pipeline.scraper:HTTP 404 on https://www.sports-reference.com/leagues/NBA_2024_games.html
INFO:src.data_pipeline.scraper:Dedup: 0 → 0 (0 removed)
INFO:src.data_pipeline.scraper:Saved 0 articles to data/raw/nba_raw.jsonl
INFO:src.data_pipeline.scraper:--- Starting NFL ---
INFO:src.data_pipeline.scraper:Fetching ESPN NFL article list from https://site.api.espn.com/apis/site/v2/sports/football/nfl/news?limit=50
INFO:src.data_pipeline.scraper:Found 50 articles in API response
INFO:src.data_pipeline.scraper:Collected 41 articles for nfl
ERROR:src.data_pipeline.scraper:HTTP 404 on https://www.sports-reference.com/years/2024/games.htm
INFO:src.data_pipeline.scraper

In [7]:
import json
from pathlib import Path

progress_file = Path("data/raw/progress.json")
with open(progress_file) as f:
    progress = json.load(f)

progress.pop("espn_nba_completed", None)

with open(progress_file, "w") as f:
    json.dump(progress, f, indent=2)

print("Cleared NBA progress")
print("Remaining keys:", list(progress.keys()))

Cleared NBA progress
Remaining keys: ['espn_nfl_completed', 'espn_mlb_completed', 'espn_nhl_completed']


In [8]:
import logging
from src.data_pipeline.scraper import SportsScraper

logging.basicConfig(level=logging.INFO)
scraper = SportsScraper(output_dir="data/raw", delay_seconds=1.0)
articles = scraper.scrape_espn_articles(sport="nba", max_articles=50)
scraper.save_to_jsonl(articles, "nba_raw.jsonl")
print(f"Saved {len(articles)} NBA articles")

INFO:src.data_pipeline.scraper:Fetching ESPN NBA article list from https://site.api.espn.com/apis/site/v2/sports/basketball/nba/news?limit=50
INFO:src.data_pipeline.scraper:Found 50 articles in API response
INFO:src.data_pipeline.scraper:Collected 42 articles for nba
INFO:src.data_pipeline.scraper:Saved 42 articles to data/raw/nba_raw.jsonl


Saved 42 NBA articles


In [15]:
pip install openai dotenv

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [dotenv]
Note: you may need to restart the kernel to use updated packages.


In [16]:
import logging
from src.data_pipeline.annotator import GPTAnnotator

from dotenv import load_dotenv
import os
load_dotenv("../.env")  # path relative to notebooks/

api_key = os.getenv("OPENAI_API_KEY")
logging.basicConfig(level=logging.INFO)

annotator = GPTAnnotator(
    raw_dir="data/raw",
    processed_dir="data/processed"
)

# Load just NBA
articles = annotator.load_raw_articles("nba_raw.jsonl")
print(f"Loaded {len(articles)} articles")

# Test sentence splitting on first article
sentences = annotator.split_into_sentences(articles[0]["body"])
print(f"Article 1 → {len(sentences)} sentences")
print(f"First sentence: {sentences[0]}")

# Annotate just 2 articles first
test_examples = annotator.annotate_articles(articles[:2])
print(f"\nGot {len(test_examples)} training examples")
print(f"\nFirst example:\n{test_examples[0]}")

INFO:src.data_pipeline.annotator:Loaded 42 articles from data/raw/nba_raw.jsonl


Loaded 42 articles
Article 1 → 11 sentences
First sentence: The 2026 NBA playoffs began Saturday, and our NBA insiders have you covered for every game in the march to the Finals. The Cleveland Cavaliers and Toronto Raptors kicked things off, and Donovan Mitchell and Co. cruised to a 13-point win at home.


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Got 16 training examples

First example:
{'text': 'The 2026 NBA playoffs began Saturday, and our NBA insiders have you covered for every game in the march to the Finals. The Cleveland Cavaliers and Toronto Raptors kicked things off, and Donovan Mitchell and Co. cruised to a 13-point win at home.', 'entities': [{'start': 4, 'end': 21, 'label': 'GAME_EVENT'}, {'start': 123, 'end': 142, 'label': 'TEAM'}, {'start': 147, 'end': 162, 'label': 'TEAM'}, {'start': 186, 'end': 202, 'label': 'PLAYER'}, {'start': 224, 'end': 232, 'label': 'STAT'}]}


In [17]:
example = test_examples[0]
text = example["text"]

print("Verifying entity offsets:\n")
for ent in example["entities"]:
    start = ent["start"]
    end = ent["end"]
    extracted = text[start:end]
    print(f"  [{ent['label']}] '{extracted}' (chars {start}-{end})")

Verifying entity offsets:

  [GAME_EVENT] '2026 NBA playoffs' (chars 4-21)
  [TEAM] 'Cleveland Cavaliers' (chars 123-142)
  [TEAM] 'Toronto Raptors' (chars 147-162)
  [PLAYER] 'Donovan Mitchell' (chars 186-202)
  [STAT] '13-point' (chars 224-232)


In [18]:
import logging
from src.data_pipeline.annotator import GPTAnnotator

logging.basicConfig(level=logging.INFO)

annotator = GPTAnnotator(
    raw_dir="data/raw",
    processed_dir="data/processed"
)

annotator.run(raw_filenames=[
    "nba_raw.jsonl",
    "nfl_raw.jsonl", 
    "mlb_raw.jsonl",
    "nhl_raw.jsonl"
])

INFO:src.data_pipeline.annotator:Annotating 4 files: ['nba_raw.jsonl', 'nfl_raw.jsonl', 'mlb_raw.jsonl', 'nhl_raw.jsonl']
INFO:src.data_pipeline.annotator:--- Annotating nba_raw.jsonl ---
INFO:src.data_pipeline.annotator:Loaded 42 articles from data/raw/nba_raw.jsonl
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO

In [5]:
pip install numpy scikit-learn

  Using cached scikit_learn-1.8.0-cp311-cp311-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.8.0-cp311-cp311-macosx_12_0_arm64.whl (8.1 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 51.0 MB/s  0:00:00m0:00:010:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [6]:
import logging
from src.data_pipeline.validator import AnnotationValidator

logging.basicConfig(level=logging.INFO)

validator = AnnotationValidator(
    processed_dir="data/processed",
    splits_dir="data/splits",
    raw_dir="data/raw"
)

report = validator.run()

print("\n=== Validation Report ===")
print(f"Total valid examples:     {report['total_examples']}")
print(f"Total entities:           {report['total_entities']}")
print(f"Avg entities per example: {report['avg_entities_per_example']}")
print(f"Flagged examples:         {report['flagged_count']}")
print(f"Weak label coverage:      {report['weak_label_coverage']['coverage_rate']:.1%}")
print(f"\nLabel distribution:")
for label, count in report['per_label_counts'].items():
    print(f"  {label:<15} {count}")

INFO:src.data_pipeline.validator:Loaded 269 examples from data/processed/mlb_annotated.jsonl
INFO:src.data_pipeline.validator:Loaded 1071 examples from data/processed/nhl_annotated.jsonl
INFO:src.data_pipeline.validator:Loaded 2628 examples from data/processed/nfl_annotated.jsonl
INFO:src.data_pipeline.validator:Loaded 3969 examples from data/processed/nba_annotated.jsonl
INFO:src.data_pipeline.validator:Total loaded: 3969 examples
INFO:src.data_pipeline.validator:Loaded 3969 total examples
INFO:src.data_pipeline.validator:Quality filter: 3967 valid, 2 flagged (0.1% flagged)
INFO:src.data_pipeline.validator:After filtering: 3967 valid examples
INFO:src.data_pipeline.validator:Weak label coverage: 93.9% (138/147)
INFO:src.data_pipeline.validator:Report: 3967 examples, 22892 entities
INFO:src.data_pipeline.validator:Label distribution: {'PLAYER': 7174, 'TEAM': 6619, 'STAT': 5157, 'POSITION': 1437, 'GAME_EVENT': 594, 'TRADE_ASSET': 568, 'INJURY': 469, 'COACH': 375, 'AWARD': 273, 'VENUE': 


=== Validation Report ===
Total valid examples:     3967
Total entities:           22892
Avg entities per example: 5.77
Flagged examples:         2
Weak label coverage:      93.9%

Label distribution:
  PLAYER          7174
  TEAM            6619
  STAT            5157
  POSITION        1437
  GAME_EVENT      594
  TRADE_ASSET     568
  INJURY          469
  COACH           375
  AWARD           273
  VENUE           226


In [9]:
import requests
r = requests.get("https://site.api.espn.com/apis/site/v2/sports/basketball/nba/news?limit=50")
data = r.json()
print(f"Articles returned: {len(data.get('articles', []))}")

Articles returned: 50


In [1]:
import logging
from src.data_pipeline.scraper import SportsScraper

logging.basicConfig(level=logging.INFO)

scraper = SportsScraper(output_dir="data/raw", delay_seconds=1.0)
articles = scraper.scrape_espn_articles(sport="nba", max_articles=50)

print(f"\nTotal articles: {len(articles)}")
print(f"Articles with full body (>300 chars): {sum(1 for a in articles if len(a['body']) > 300)}")
print(f"Articles falling back to description: {sum(1 for a in articles if a['body'] == a['description'])}")

ModuleNotFoundError: No module named 'src'

## Load Raw Articles

In [ ]:
raw_dir = Path('../data/raw')

# TODO: Load all JSONL files from raw_dir into a DataFrame.
# TODO: Show shape, dtypes, and sample rows.

## Article Length Distribution

In [ ]:
# TODO: Plot histogram of article body lengths (characters and tokens).
# TODO: Annotate with median and 95th percentile.

## Source & Sport Distribution

In [ ]:
# TODO: Bar chart of article counts per sport and per source.
# TODO: Flag any imbalance that may require upsampling.

## Vocabulary Analysis

In [ ]:
# TODO: Compute top-100 tokens by frequency.
# TODO: Identify sports-specific terms that should be well-covered by DeBERTa-v3.